# Chores

In [114]:
pip install opencv-python pandas tqdm matplotlib

  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ----------- ---------------------------- 2.4/8.2 MB 11.2 MB/s eta 0:00:01
   ------------------------ --------------- 5.0/8.2 MB 11.6 MB/s eta 0:00:01
   ----------------------------------- ---- 7.3/8.2 MB 11.9 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 11.6 MB/s  0:00:00
Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ----------------------------------- ---- 2.1/2.3 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 11.0 MB/s  0:00:00
   ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
   ------------- -------------------------- 2.4/7.1 MB 11.2 MB/s eta 0:00:01
   -

In [91]:
import cv2 as cv
import numpy as np
import os
import pandas as pd
import shutil

In [115]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [93]:
base_path = os.path.join(".", "paris", "data", "img")
dirs = os.listdir(base_path)

for dir in dirs:
  for file in os.listdir(os.path.join(base_path, dir)):
    src = os.path.join(base_path, dir, file)
    print(src)
    dst = base_path
    shutil.move(src, dst)

NotADirectoryError: [WinError 267] The directory name is invalid: '.\\paris\\data\\img\\desktop.ini'

In [ ]:
for dir in dirs:
  os.removedirs(os.path.join(base_path, dir))

In [ ]:
from pathlib import Path

def tree(path=Path("."), depth=3, prefix=""):
    if depth < 0:
        return

    entries = sorted(path.iterdir(), key=lambda p: p.name)

    for i, p in enumerate(entries):
        # Skip hidden directories
        if p.is_dir() and p.name.startswith("."):
            continue

        connector = "└── " if i == len(entries) - 1 else "├── "
        print(prefix + connector + p.name)

        if p.is_dir():
            extension = "    " if i == len(entries) - 1 else "│   "
            tree(p, depth - 1, prefix + extension)

tree(Path("."), depth=2)

├── local-desc.ipynb
├── oxford
│   ├── compute_ap.cpp
│   └── data
│       ├── gt
│       └── img
├── paris
│   └── data
│       ├── gt
│       └── img
├── requirements.txt
└── zips
    ├── gt_files_170407.tgz
    ├── oxbuild_images.tgz.zip
    ├── paris_1.tgz
    ├── paris_120310.tgz
    └── paris_2.tgz


# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from src.paths import ensure_dirs, RESULTS_DIR
from src.features import extract_or_load_sift_meanstd
from src.experiments import generate_rankings
from src.evaluation import compile_compute_ap, evaluate_experiment

import pandas as pd

ensure_dirs()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Feature extraction / load

In [94]:
oxford_ids, oxford_X = extract_or_load_sift_meanstd("oxford")
paris_ids, paris_X = extract_or_load_sift_meanstd("paris")

# oxford_ids, oxford_X = extract_or_load_sift_meanstd("oxford", force=True)
# paris_ids, paris_X = extract_or_load_sift_meanstd("paris", force=True)

Loaded cached oxford: (5063, 256)
Loaded cached paris: (6392, 256)


## Removal of corrupt images

In [ ]:
# to_remove = os.path.join(".", "paris", "data", "img", "paris_louvre_000136.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_louvre_000146.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_moulinrouge_000422.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_museedorsay_001059.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_notredame_000188.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pantheon_000284.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pantheon_000960.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pantheon_000974.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pompidou_000195.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pompidou_000196.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pompidou_000201.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pompidou_000467.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pompidou_000640.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_sacrecoeur_000299.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_sacrecoeur_000330.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_sacrecoeur_000353.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_triomphe_000662.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_triomphe_000833.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_triomphe_000863.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_triomphe_000867.jpg")
# os.remove(to_remove)

# Ranking

In [ ]:
DATASETS = ["oxford", "paris"]
METHODS = ["sift_meanstd"]
METRICS = ["cosine", "euclidean"]

In [106]:
from itertools import product

EXP_NAMES = [a + "_" + b for a,b in zip(METHODS, METRICS)]
EXP_NAMES_BBOX = [a + "_bbox_" + b for a,b in list(product(METHODS, METRICS))]

In [ ]:
for dataset, ids, X in [
    ("oxford", oxford_ids, oxford_X),
    ("paris", paris_ids, paris_X),
]:
    for metric in METRICS:
        exp_name, out_dir = generate_rankings(dataset, ids, X, metric)
        print(dataset, exp_name, out_dir)

Ranking oxford/cosine/bbox: 100%|██████████| 55/55 [00:04<00:00, 11.74it/s]


oxford sift_meanstd_bbox_cosine C:\Users\iansg\code\cv\P1\results\rankings\oxford\sift_meanstd_bbox_cosine


Ranking oxford/euclidean/bbox: 100%|██████████| 55/55 [00:04<00:00, 12.36it/s]


oxford sift_meanstd_bbox_euclidean C:\Users\iansg\code\cv\P1\results\rankings\oxford\sift_meanstd_bbox_euclidean


Ranking paris/cosine/bbox: 100%|██████████| 55/55 [00:04<00:00, 12.52it/s]


paris sift_meanstd_bbox_cosine C:\Users\iansg\code\cv\P1\results\rankings\paris\sift_meanstd_bbox_cosine


Ranking paris/euclidean/bbox: 100%|██████████| 55/55 [00:04<00:00, 13.23it/s]

paris sift_meanstd_bbox_euclidean C:\Users\iansg\code\cv\P1\results\rankings\paris\sift_meanstd_bbox_euclidean


# Evaluation

In [96]:
compute_ap_exe = compile_compute_ap()
compute_ap_exe

WindowsPath('C:/Users/iansg/code/cv/P1/compute_ap')

In [108]:
all_rows = []

for dataset in DATASETS:
    for exp_name in EXP_NAMES_BBOX:
        df = evaluate_experiment(dataset, exp_name, compute_ap_exe)
        all_rows.append(df)

ap_df = pd.concat(all_rows, ignore_index=True)
ap_df.head()

Evaluating paris/sift_meanstd_bbox_euclidean: 100%|██████████| 55/55 [00:00<00:00, 147.20it/s]


,dataset,experiment,query,ap
0,oxford,sift_meanstd_bbox_cosine,all_souls_1,0.083817
1,oxford,sift_meanstd_bbox_cosine,all_souls_2,0.060034
2,oxford,sift_meanstd_bbox_cosine,all_souls_3,0.100546
3,oxford,sift_meanstd_bbox_cosine,all_souls_4,0.108560
4,oxford,sift_meanstd_bbox_cosine,all_souls_5,0.128537


## Summary

In [109]:
ap_df.to_csv(RESULTS_DIR / "ap_per_query.csv", index=False)

summary = (
    ap_df
    .groupby(["dataset", "experiment"], as_index=False)
    .agg(map=("ap", "mean"), num_queries=("ap", "count"))
    .sort_values("map", ascending=False)
)

summary.to_csv(RESULTS_DIR / "summary_sorted.csv", index=False)
summary

,dataset,experiment,map,num_queries
2,paris,sift_meanstd_bbox_cosine,0.174703,55
3,paris,sift_meanstd_bbox_euclidean,0.174700,55
0,oxford,sift_meanstd_bbox_cosine,0.119246,55
1,oxford,sift_meanstd_bbox_euclidean,0.119241,55


dataset                           paris
experiment     sift_meanstd_bbox_cosine
map                            0.174703
num_queries                          55
Name: 2, dtype: object


In [112]:
print(len(summary))

4


## Visualization

### Top 5

In [ ]:
from src.visualization import save_all_top5_visualizations

best = summary.iloc[0]
# worst = summary.iloc[len(summary)-1]

paths = save_all_top5_visualizations(
    dataset=best["dataset"],
    exp_name=best["experiment"]
)

paths[:3]

paris
sift_meanstd_bbox_cosine


[WindowsPath('C:/Users/iansg/code/cv/P1/results/qualitative/paris/sift_meanstd_bbox_cosine/defense_1.png'),
 WindowsPath('C:/Users/iansg/code/cv/P1/results/qualitative/paris/sift_meanstd_bbox_cosine/defense_2.png'),
 WindowsPath('C:/Users/iansg/code/cv/P1/results/qualitative/paris/sift_meanstd_bbox_cosine/defense_3.png')]

### Success | Median | Failure examples

In [119]:
from src.visualization import save_success_failure_visualizations

cases_df = save_success_failure_visualizations()
cases_df

,dataset,experiment,case,query,ap,visualization
0,oxford,sift_meanstd_bbox_cosine,worst,pitt_rivers_3,0.001118,C:\Users\iansg\code\cv\P1\results\qualitative\...
1,oxford,sift_meanstd_bbox_cosine,median,hertford_3,0.077291,C:\Users\iansg\code\cv\P1\results\qualitative\...
2,oxford,sift_meanstd_bbox_cosine,best,keble_1,0.694218,C:\Users\iansg\code\cv\P1\results\qualitative\...
3,oxford,sift_meanstd_bbox_euclidean,worst,pitt_rivers_3,0.001118,C:\Users\iansg\code\cv\P1\results\qualitative\...
4,oxford,sift_meanstd_bbox_euclidean,median,hertford_3,0.077278,C:\Users\iansg\code\cv\P1\results\qualitative\...
5,oxford,sift_meanstd_bbox_euclidean,best,keble_1,0.694218,C:\Users\iansg\code\cv\P1\results\qualitative\...
6,paris,sift_meanstd_bbox_cosine,worst,museedorsay_2,0.013089,C:\Users\iansg\code\cv\P1\results\qualitative\...
7,paris,sift_meanstd_bbox_cosine,median,triomphe_4,0.142572,C:\Users\iansg\code\cv\P1\results\qualitative\...
8,paris,sift_meanstd_bbox_cosine,best,pompidou_3,0.690029,C:\Users\iansg\code\cv\P1\results\qualitative\...
9,paris,sift_meanstd_bbox_euclidean,worst,museedorsay_2,0.013089,C:\Users\iansg\code\cv\P1\results\qualitative\...


## Inspection

In [89]:
from pathlib import Path

sample = next((RESULTS_DIR / "rankings" / "oxford" / "sift_meanstd_cosine").glob("*.txt"))

print(sample)

with open(sample) as f:
    for _ in range(10):
        print(f.readline().strip())

C:\Users\iansg\code\cv\P1\results\rankings\oxford\sift_meanstd_cosine\all_souls_1.txt
christ_church_000757
all_souls_000015
christ_church_000093
oxford_003587
radcliffe_camera_000447
radcliffe_camera_000160
christ_church_000396
radcliffe_camera_000164
radcliffe_camera_000036
magdalen_000995
